> **Note for teachers:** the `assert` cells are CI self-checks that students can ignore.
> This is one worked reference, not the answer: any point on the frontier is acceptable if it is measured
> and defended (see the rubric in `teacher-notes.md`).
> Every number is measured on the `val` split of the main log; the sealed final split is never opened.
> The memo answers are built with f-strings from the measured tables, so every number in the prose is the
> number in the table above it.

# Capstone reference: a reach-first library recommender

This reference builds a four-path blend (item-item, two-tower, feature tower and semantic), drops the learned
reranker, and defends a point on the frontier that gives up a little hit for a much wider shelf.

## Setup — load the whole system

The setup cell is the brief's, unchanged: it loads the model cache, assembles the system, and builds
`zero_train_books`.

In [1]:
from bookrec import FullSystem, load_cached_models
from bookrec.system import FULL8_PATH_NAMES, POOL, FixedBlendPath

K = 10

cache = load_cached_models()          # fails with the build command if the cache is missing
system = FullSystem.from_cache(cache)
data = system.data

trained_books = {
    row["item_id"] for row in data.rows if row["split"] == "train" and row["label"] == 1
}
zero_train_books = set(data.catalog_ids) - trained_books


def cold_surfaced(lists):
    # Distinct zero-train books that appear in any reader's top-10.
    return len({book for top in lists.values() for book in top if book in zero_train_books})


def show(name, metrics):
    print(
        f"{name:<24} hit@10 {metrics['hit']:.3f}  coverage {metrics['coverage']:.3f}  "
        f"head-share {metrics['head_share']:.3f}  Gini {metrics['exposure_gini']:.3f}"
    )


print(f"{len(data.cohort)} val readers, {len(data.catalog_ids)} books, "
      f"{len(zero_train_books)} books with no train positive")
print("the eight served paths:", list(FULL8_PATH_NAMES))

500 val readers, 2000 books, 818 books with no train positive
the eight served paths: ['popularity', 'lexical', 'item-item', 'mf', 'semantic', 'two-tower', 'feature-tower', 'hybrid']


In [2]:
assert len(data.cohort) == 500
assert len(FULL8_PATH_NAMES) == 8 and set(FULL8_PATH_NAMES) == set(system.full8)
assert 0 < len(zero_train_books) < len(data.catalog_ids)

## Milestone 1

### Choose your retrieval paths, and say why

Every path is scored alone as a one-path blend over its top-50 pool.
The table then drives the choice: one path for reach, one for hit, one for both hit and cold books, and one
for cold books.

In [3]:
path_table = []
for name in FULL8_PATH_NAMES:
    single = FixedBlendPath({name: system.full8[name]}, pool=POOL, name=name)
    lists = system.top_lists(single)
    metrics = system.readout(lists)
    cold = cold_surfaced(lists)
    path_table.append(
        {"path": name, "hit": metrics["hit"], "coverage": metrics["coverage"], "cold_books": cold}
    )
    show(name, metrics)
    print(f"{'':<24} cold books shown {cold}")

by_path = {row["path"]: row for row in path_table}

chosen_paths = ["item-item", "two-tower", "feature-tower", "semantic"]

ii, tt = by_path["item-item"], by_path["two-tower"]
ft, sem = by_path["feature-tower"], by_path["semantic"]
path_reasons = {
    "item-item": (
        f"Reach: its {ii['coverage']:.3f} coverage is the widest of all eight paths, and it still "
        f"hits {ii['hit']:.3f}."
    ),
    "two-tower": (
        f"Hit: {tt['hit']:.3f} hit@10 is the strongest pure collaborative signal; I left out the hybrid "
        f"({by_path['hybrid']['hit']:.3f}) because it re-uses this same two-tower."
    ),
    "feature-tower": (
        f"Hit and cold books: {ft['hit']:.3f} hit@10 with {ft['coverage']:.3f} coverage, and it is "
        f"the only learned path that shows zero-train books ({ft['cold_books']})."
    ),
    "semantic": (
        f"Cold books: it shows {sem['cold_books']} zero-train books, the most of any path, at "
        f"{sem['coverage']:.3f} coverage, even though its own hit is only {sem['hit']:.3f}."
    ),
}
print()
for name in chosen_paths:
    print(f"{name}: {path_reasons[name]}")

popularity               hit@10 0.108  coverage 0.008  head-share 1.000  Gini 0.995
                         cold books shown 0


lexical                  hit@10 0.158  coverage 0.278  head-share 0.268  Gini 0.888
                         cold books shown 128


item-item                hit@10 0.252  coverage 0.490  head-share 0.491  Gini 0.773
                         cold books shown 0


mf                       hit@10 0.276  coverage 0.074  head-share 0.994  Gini 0.971
                         cold books shown 0


semantic                 hit@10 0.102  coverage 0.354  head-share 0.109  Gini 0.857
                         cold books shown 263


two-tower                hit@10 0.340  coverage 0.177  head-share 0.900  Gini 0.929
                         cold books shown 0


feature-tower            hit@10 0.298  coverage 0.306  head-share 0.638  Gini 0.877
                         cold books shown 109


hybrid                   hit@10 0.360  coverage 0.192  head-share 0.864  Gini 0.922
                         cold books shown 17

item-item: Reach: its 0.490 coverage is the widest of all eight paths, and it still hits 0.252.
two-tower: Hit: 0.340 hit@10 is the strongest pure collaborative signal; I left out the hybrid (0.360) because it re-uses this same two-tower.
feature-tower: Hit and cold books: 0.298 hit@10 with 0.306 coverage, and it is the only learned path that shows zero-train books (109).
semantic: Cold books: it shows 263 zero-train books, the most of any path, at 0.354 coverage, even though its own hit is only 0.102.


In [4]:
assert len(path_table) == 8
assert all(row[key] is not None for row in path_table for key in ("hit", "coverage", "cold_books"))
assert len(chosen_paths) >= 3 and set(chosen_paths) <= set(FULL8_PATH_NAMES)
assert set(path_reasons) == set(chosen_paths)
# The claims in the reasons hold on the measured table:
assert max(path_table, key=lambda row: row["coverage"])["path"] == "item-item"
assert max(path_table, key=lambda row: row["cold_books"])["path"] == "semantic"
assert ft["cold_books"] > 0 and tt["cold_books"] == 0 and by_path["mf"]["cold_books"] == 0

## Milestone 2

### Build the candidate pool and the blend

First the four paths at equal weight, then one change: item-item raised to 1.5, to lean the blend toward reach.
The weighted version is the one I keep as `my_blend`.

In [5]:
start_weights = {name: 1.0 for name in chosen_paths}
start_blend = FixedBlendPath(
    {n: system.full8[n] for n in chosen_paths}, weights=start_weights, pool=POOL, name="equal-blend"
)
start_lists = system.top_lists(start_blend)
start_metrics = system.readout(start_lists)
start_cold = cold_surfaced(start_lists)

my_weights = {"item-item": 1.5, "two-tower": 1.0, "feature-tower": 1.0, "semantic": 1.0}

my_blend = FixedBlendPath(
    {n: system.full8[n] for n in chosen_paths}, weights=my_weights, pool=POOL, name="my-blend"
)
my_lists = system.top_lists(my_blend)
my_metrics = system.readout(my_lists)
my_cold = cold_surfaced(my_lists)

fixed_blend = system.fixed_blend()
fixed_metrics = system.evaluate(fixed_blend)

show("equal-weight 4-path", start_metrics)
show("my-blend (item-item 1.5)", my_metrics)
show("8-path fixed blend", fixed_metrics)
print(f"cold books shown: equal-weight {start_cold}, my-blend {my_cold}")

blend_reading = (
    f"My blend scores {my_metrics['hit']:.3f} hit@10 against the 8-path blend's "
    f"{fixed_metrics['hit']:.3f}, so it gives up {fixed_metrics['hit'] - my_metrics['hit']:.3f} "
    f"(about {round((fixed_metrics['hit'] - my_metrics['hit']) * len(data.cohort))} readers), "
    f"but its coverage is {my_metrics['coverage']:.3f} against {fixed_metrics['coverage']:.3f}. "
    f"Raising item-item from 1.0 to 1.5 moved the blend from {start_metrics['hit']:.3f} / "
    f"{start_metrics['coverage']:.3f} to {my_metrics['hit']:.3f} / {my_metrics['coverage']:.3f} "
    f"(hit / coverage), better on both, but cold books shown fell from {start_cold} to {my_cold}."
)
print()
print(blend_reading)

equal-weight 4-path      hit@10 0.340  coverage 0.318  head-share 0.758  Gini 0.887
my-blend (item-item 1.5) hit@10 0.352  coverage 0.376  head-share 0.726  Gini 0.867
8-path fixed blend       hit@10 0.362  coverage 0.206  head-share 0.846  Gini 0.923
cold books shown: equal-weight 56, my-blend 29

My blend scores 0.352 hit@10 against the 8-path blend's 0.362, so it gives up 0.010 (about 5 readers), but its coverage is 0.376 against 0.206. Raising item-item from 1.0 to 1.5 moved the blend from 0.340 / 0.318 to 0.352 / 0.376 (hit / coverage), better on both, but cold books shown fell from 56 to 29.


In [6]:
assert my_weights.keys() == set(chosen_paths) and start_weights.keys() == set(chosen_paths)
assert my_weights != start_weights
assert len(my_lists) == len(data.cohort) and all(len(top) == K for top in my_lists.values())
# The reading's claims: a trade against the 8-path blend (less hit, more reach) ...
assert my_metrics["hit"] < fixed_metrics["hit"]
assert my_metrics["coverage"] > fixed_metrics["coverage"] + 0.1
# ... and the weight change helped hit and coverage but cost cold books.
assert my_metrics["hit"] > start_metrics["hit"] and my_metrics["coverage"] > start_metrics["coverage"]
assert my_cold < start_cold

## Milestone 3

### Add the learned reranker, and decide whether it stays

The reranker's training record comes from the cache; the full stack is scored on the same cohort.

In [7]:
reranker_loss = cache.loss_history("reranker-full8")
first_loss = reranker_loss[0]
last_loss = reranker_loss[-1]
n_features = cache.meta("reranker-full8")["n_features"]
training_rows = cache.meta("reranker-full8")["training_rows"]
print(f"reranker-full8: {len(reranker_loss)} epochs, loss {first_loss:.3f} -> {last_loss:.3f}, "
      f"{n_features} features, {training_rows} training rows")

full_stack_lists = system.top_lists(system.full_stack)
full_stack_metrics = system.readout(full_stack_lists)
full_stack_cold = cold_surfaced(full_stack_lists)

fixed_cold = cold_surfaced(system.top_lists(fixed_blend))
show("full stack (reranked)", full_stack_metrics)
show("8-path fixed blend", fixed_metrics)
show("my-blend", my_metrics)
print(f"cold books shown: full stack {full_stack_cold}, fixed blend {fixed_cold}, my-blend {my_cold}")

keep_reranker = full_stack_metrics["hit"] > fixed_metrics["hit"] and full_stack_cold > 0
final_system = system.full_stack if keep_reranker else my_blend
final_lists = full_stack_lists if keep_reranker else my_lists
final_metrics = system.readout(final_lists)

reranker_reading = (
    f"The reranker trained cleanly (loss {first_loss:.3f} to {last_loss:.3f} over {training_rows} "
    f"rows), yet the full stack scores {full_stack_metrics['hit']:.3f} hit@10, below the fixed "
    f"blend's {fixed_metrics['hit']:.3f} over the same pool, with coverage "
    f"{full_stack_metrics['coverage']:.3f} against {fixed_metrics['coverage']:.3f}. "
    f"It shows {full_stack_cold} zero-train books where the fixed blend shows {fixed_cold} and my "
    f"blend {my_cold}: it learned from warm histories and pushes every cold book out. "
    f"It costs hit, reach and cold books at once, so I drop it and ship my blend."
)
print()
print(f"keep_reranker = {keep_reranker}")
print(reranker_reading)

reranker-full8: 30 epochs, loss 0.412 -> 0.229, 21 features, 7458 training rows


full stack (reranked)    hit@10 0.356  coverage 0.203  head-share 0.847  Gini 0.913
8-path fixed blend       hit@10 0.362  coverage 0.206  head-share 0.846  Gini 0.923
my-blend                 hit@10 0.352  coverage 0.376  head-share 0.726  Gini 0.867
cold books shown: full stack 0, fixed blend 23, my-blend 29

keep_reranker = False
The reranker trained cleanly (loss 0.412 to 0.229 over 7458 rows), yet the full stack scores 0.356 hit@10, below the fixed blend's 0.362 over the same pool, with coverage 0.203 against 0.206. It shows 0 zero-train books where the fixed blend shows 23 and my blend 29: it learned from warm histories and pushes every cold book out. It costs hit, reach and cold books at once, so I drop it and ship my blend.


In [8]:
assert n_features == 21 and training_rows > 0
assert last_loss < first_loss
assert full_stack_cold == 0 and my_cold > 0
assert full_stack_metrics["hit"] < fixed_metrics["hit"]
assert keep_reranker is False and final_system is my_blend and final_lists is my_lists
assert final_metrics == my_metrics

## Milestone 4

### The ablation table: what does each piece buy?

My path rows compare with **my blend**; the reranker stage row compares the full stack with the **8-path fixed
blend**.

In [9]:
my_ablation = []
for dropped in chosen_paths:
    kept = {n: system.full8[n] for n in chosen_paths if n != dropped}
    kept_weights = {n: w for n, w in my_weights.items() if n != dropped}
    without = FixedBlendPath(kept, weights=kept_weights, pool=POOL, name=f"minus-{dropped}")
    without_lists = system.top_lists(without)
    metrics = system.readout(without_lists)
    my_ablation.append({
        "row": f"minus-{dropped}",
        "baseline": "my-blend",
        "d_hit": metrics["hit"] - my_metrics["hit"],
        "d_coverage": metrics["coverage"] - my_metrics["coverage"],
        "cold_books": cold_surfaced(without_lists),
    })

my_ablation.append({
    "row": "add-reranker",
    "baseline": "fixed-blend",
    "d_hit": full_stack_metrics["hit"] - fixed_metrics["hit"],
    "d_coverage": full_stack_metrics["coverage"] - fixed_metrics["coverage"],
})

reference = cache.readouts
print("reference baselines:")
for name, base in reference["baselines"].items():
    print(f"  {name:<12} hit@10 {base['hit']:.3f}  coverage {base['coverage']:.3f}")
print("reference ablation (row - baseline):")
for row in reference["ablation"]:
    print(f"  {row['row']:<22} vs {row['baseline']:<12} d_hit {row['d_hit']:+.3f}  "
          f"d_coverage {row['d_coverage']:+.3f}")
print("my ablation (row - baseline):")
for row in my_ablation:
    print(f"  {row['row']:<22} vs {row['baseline']:<12} d_hit {row['d_hit']:+.3f}  "
          f"d_coverage {row['d_coverage']:+.3f}  cold books {row.get('cold_books', '-')}")

mine = {row["row"]: row for row in my_ablation}
cold_minus_semantic = mine["minus-semantic"]["cold_books"]
ref = {row["row"]: row for row in reference["ablation"]}
ablation_reading = (
    f"The two-tower matters most: dropping it costs {-mine['minus-two-tower']['d_hit']:.3f} hit "
    f"(it would also add {mine['minus-two-tower']['d_coverage']:+.3f} coverage), and dropping "
    f"item-item costs {-mine['minus-item-item']['d_hit']:.3f} hit and "
    f"{-mine['minus-item-item']['d_coverage']:.3f} coverage. Semantic barely matters for hit "
    f"({mine['minus-semantic']['d_hit']:+.3f}, about "
    f"{round(-mine['minus-semantic']['d_hit'] * len(data.cohort))} readers), but it is where my cold books "
    f"come from. No removal raises hit in my blend, but in the reference table dropping popularity "
    f"raises the 8-path blend's hit by {ref['minus-popularity']['d_hit']:+.3f}, and dropping the "
    f"reranker raises the full stack's by {ref['no-reranker']['d_hit']:+.3f}."
)
print()
print(ablation_reading)

reference baselines:
  full-stack   hit@10 0.356  coverage 0.203
  fixed-blend  hit@10 0.362  coverage 0.206
reference ablation (row - baseline):
  no-reranker            vs full-stack   d_hit +0.006  d_coverage +0.003
  exact-two-tower        vs full-stack   d_hit -0.002  d_coverage +0.000
  minus-popularity       vs fixed-blend  d_hit +0.008  d_coverage +0.014
  minus-lexical          vs fixed-blend  d_hit +0.000  d_coverage -0.031
  minus-item-item        vs fixed-blend  d_hit -0.008  d_coverage -0.014
  minus-mf               vs fixed-blend  d_hit -0.008  d_coverage +0.032
  minus-semantic         vs fixed-blend  d_hit -0.004  d_coverage -0.020
  minus-two-tower        vs fixed-blend  d_hit -0.010  d_coverage +0.015
  minus-feature-tower    vs fixed-blend  d_hit +0.002  d_coverage +0.002
  minus-hybrid           vs fixed-blend  d_hit -0.006  d_coverage +0.005
my ablation (row - baseline):
  minus-item-item        vs my-blend     d_hit -0.026  d_coverage -0.092  cold books 118
  min

In [10]:
assert len(my_ablation) == len(chosen_paths) + 1
assert all(row["baseline"] in ("my-blend", "fixed-blend") for row in my_ablation)
assert len(reference["ablation"]) == 10
assert abs(mine["add-reranker"]["d_hit"] - (full_stack_metrics["hit"] - fixed_metrics["hit"])) < 1e-12
# The reading's claims hold on the measured rows:
path_rows = my_ablation[:-1]
assert min(path_rows, key=lambda row: row["d_hit"])["row"] == "minus-two-tower"
assert all(row["d_hit"] <= 0 for row in path_rows)
assert abs(mine["minus-semantic"]["d_hit"]) <= 0.01
assert ref["minus-popularity"]["d_hit"] > 0 and ref["no-reranker"]["d_hit"] > 0

## Milestone 5

### Beyond accuracy, and the Part-1 benchmark

Four systems, six columns, plus each system's cold-book count.

In [11]:
best_single_lists = system.top_lists(system.best_single_path)
best_single_metrics = system.readout(best_single_lists)
part1_lists = system.top_lists(system.part1_build())
part1_metrics = system.readout(part1_lists)

beyond_rows = [
    {"system": "my-final-system", **final_metrics, "cold_books": my_cold},
    {"system": "full-stack", **full_stack_metrics, "cold_books": full_stack_cold},
    {"system": "best-single-path", **best_single_metrics,
     "cold_books": cold_surfaced(best_single_lists)},
    {"system": "part1-build", **part1_metrics, "cold_books": cold_surfaced(part1_lists)},
]
columns = ["hit", "coverage", "head_share", "exposure_gini", "novelty", "diversity"]

print(f"{'system':<18}" + "".join(f"{c:>15}" for c in columns) + f"{'cold_books':>12}")
for row in beyond_rows:
    print(f"{row['system']:<18}" + "".join(f"{row[c]:>15.3f}" for c in columns)
          + f"{row['cold_books']:>12}")
print()
print("the build's own table:")
for row in cache.readouts["beyond_accuracy"]:
    print(f"{row['system']:<18}" + "".join(f"{row[c]:>15.3f}" for c in columns))

benchmark_reading = (
    f"My system beats the Part-1 build on hit ({final_metrics['hit']:.3f} vs "
    f"{part1_metrics['hit']:.3f}), on reach (coverage {final_metrics['coverage']:.3f} vs "
    f"{part1_metrics['coverage']:.3f}) and on exposure (head-share {final_metrics['head_share']:.3f} "
    f"vs {part1_metrics['head_share']:.3f}, Gini {final_metrics['exposure_gini']:.3f} vs "
    f"{part1_metrics['exposure_gini']:.3f}), and on novelty too. "
    f"It still loses on diversity ({final_metrics['diversity']:.3f} vs "
    f"{part1_metrics['diversity']:.3f}), and so does every other system in the table, so nothing "
    f"here beats the Part-1 build on every column; the full stack and the best single path beat it "
    f"only on hit and lose on coverage and head-share."
)
print()
print(benchmark_reading)

system                        hit       coverage     head_share  exposure_gini        novelty      diversity  cold_books
my-final-system             0.352          0.376          0.726          0.867          8.931          0.851          29
full-stack                  0.356          0.203          0.847          0.913          8.568          0.857           0
best-single-path            0.362          0.192          0.862          0.922          8.483          0.850          17
part1-build                 0.306          0.333          0.793          0.908          8.492          0.871           4

the build's own table:
full-system                 0.356          0.203          0.847          0.913          8.568          0.857
best-single-path            0.362          0.192          0.862          0.922          8.483          0.850
part1-build                 0.306          0.333          0.793          0.908          8.492          0.871

My system beats the Part-1 build on hit (0.

In [12]:
assert [row["system"] for row in beyond_rows] == [
    "my-final-system", "full-stack", "best-single-path", "part1-build"
]
assert all(c in row for row in beyond_rows for c in columns)
# My system beats Part 1 on hit, reach and both exposure columns ...
assert final_metrics["hit"] > part1_metrics["hit"]
assert final_metrics["coverage"] > part1_metrics["coverage"]
assert final_metrics["head_share"] < part1_metrics["head_share"]
assert final_metrics["exposure_gini"] < part1_metrics["exposure_gini"]
assert final_metrics["novelty"] > part1_metrics["novelty"]
# ... but no system in the table beats it on diversity, and the two hit leaders lose on reach.
assert all(row["diversity"] < part1_metrics["diversity"] for row in beyond_rows[:3])
assert full_stack_metrics["coverage"] < part1_metrics["coverage"]
assert best_single_metrics["coverage"] < part1_metrics["coverage"]

## Milestone 6

### The design memo: defend your point on the frontier

My system needs item-item and semantic (refit in seconds as cheap paths), the two-tower served through the HNSW
index, and the feature tower.

In [13]:
build_seconds = cache.manifest["timings_s"]
print("cache build times (s):", build_seconds)
needs = {
    "item-item, semantic (cheap paths)": build_seconds["cheap-paths"],
    "two-tower": build_seconds["two-tower"],
    "HNSW index (ann)": build_seconds["ann"],
    "feature-tower": build_seconds["feature-tower"],
}
for model, seconds in needs.items():
    print(f"  my system needs {model:<34} {seconds:>6.1f} s")
my_build = sum(needs.values())
full_stack_build = (
    build_seconds["cheap-paths"] + build_seconds["mf"] + build_seconds["two-tower"]
    + build_seconds["ann"] + build_seconds["feature-tower"] + build_seconds["reranker-full8"]
)
print(f"my system: {my_build:.1f} s to build; full stack: {full_stack_build:.1f} s")

hit_gap = fixed_metrics["hit"] - final_metrics["hit"]
frontier_choice = (
    f"I put the system at the reach end of the frontier: {final_metrics['hit']:.3f} hit@10 with "
    f"{final_metrics['coverage']:.3f} coverage, a head-share of {final_metrics['head_share']:.3f} and "
    f"an exposure Gini of {final_metrics['exposure_gini']:.3f}. "
    f"The 8-path fixed blend beats it on hit ({fixed_metrics['hit']:.3f}), but only by "
    f"{hit_gap:.3f}, about {round(hit_gap * len(data.cohort))} of {len(data.cohort)} readers, while "
    f"showing {fixed_metrics['coverage']:.3f} of the catalog. "
    f"A library's front page is how readers find the shelf, so reaching "
    f"{final_metrics['coverage'] / fixed_metrics['coverage']:.1f} times as many books is worth a "
    f"handful of readers' hits."
)
accuracy_vs_reach = (
    f"Against the highest-hit systems I gave up {hit_gap:.3f} hit to gain "
    f"{final_metrics['coverage'] - fixed_metrics['coverage']:.3f} coverage and cut head-share from "
    f"{fixed_metrics['head_share']:.3f} to {final_metrics['head_share']:.3f}. "
    f"Against the Part-1 build I gave up nothing: mine is ahead of it on hit by "
    f"{final_metrics['hit'] - part1_metrics['hit']:.3f} and on coverage by "
    f"{final_metrics['coverage'] - part1_metrics['coverage']:.3f}. "
    f"My own ablation shows the knob: dropping the two-tower would add "
    f"{mine['minus-two-tower']['d_coverage']:+.3f} coverage but cost "
    f"{-mine['minus-two-tower']['d_hit']:.3f} hit, a trade I judged too steep."
)
cold_start = (
    f"My system shows {my_cold} of the {len(zero_train_books)} zero-train books; the full stack "
    f"shows {full_stack_cold}. "
    f"Almost all of mine come from the semantic path: without it the blend shows "
    f"only {cold_minus_semantic}. Item-item and the two-tower cannot score a book nobody has read; the "
    f"feature tower can, but in my blend its cold scores lose to warm books almost every time. "
    f"So a new author's first book can reach the front page if its description sounds like what a "
    f"reader already likes, but {my_cold} books is still a small door, and the equal-weight version "
    f"of my blend opened it wider ({start_cold})."
)
cost = (
    f"My system needs about {my_build:.0f} s of model building (the feature tower "
    f"{build_seconds['feature-tower']:.1f} s and the two-tower {build_seconds['two-tower']:.1f} s "
    f"are most of it); the full stack needs about {full_stack_build:.0f} s, with the reranker alone "
    f"at {build_seconds['reranker-full8']:.1f} s, plus scoring 21 features for every candidate at "
    f"serving time. "
    f"I cut the reranker first, and the reference ablation says that loses nothing: removing it "
    f"changes hit by {ref['no-reranker']['d_hit']:+.3f} and coverage by "
    f"{ref['no-reranker']['d_coverage']:+.3f}. "
    f"Within my own system the next cut would be the feature tower, which my ablation says costs "
    f"{-mine['minus-feature-tower']['d_hit']:.3f} hit (while adding "
    f"{mine['minus-feature-tower']['d_coverage']:+.3f} coverage)."
)
next_step = (
    f"I would try an exploration slot from Unit 13: keep my blend for nine places and give the "
    f"tenth to the best-scoring zero-train book from the semantic path. "
    f"It works if the zero-train books shown rise well above {my_cold} while hit@10 stays at or "
    f"above {start_metrics['hit']:.3f} (my equal-weight blend) and coverage stays above "
    f"{part1_metrics['coverage']:.3f} (the Part-1 build), all on val."
)
for title, text in [("frontier_choice", frontier_choice), ("accuracy_vs_reach", accuracy_vs_reach),
                    ("cold_start", cold_start), ("cost", cost), ("next_step", next_step)]:
    print(f"\n{title}:\n{text}")

cache build times (s): {'cheap-paths': 0.2, 'mf': 16.7, 'two-tower': 17.5, 'feature-tower': 26.3, 'feature-tower-hard': 26.7, 'ann': 0.2, 'reranker-u11': 36.9, 'reranker-full8': 65.2, 'sasrec': 45.2, 'readouts': 54.7, 'total': 289.8}
  my system needs item-item, semantic (cheap paths)     0.2 s
  my system needs two-tower                            17.5 s
  my system needs HNSW index (ann)                      0.2 s
  my system needs feature-tower                        26.3 s
my system: 44.2 s to build; full stack: 126.1 s

frontier_choice:
I put the system at the reach end of the frontier: 0.352 hit@10 with 0.376 coverage, a head-share of 0.726 and an exposure Gini of 0.867. The 8-path fixed blend beats it on hit (0.362), but only by 0.010, about 5 of 500 readers, while showing 0.206 of the catalog. A library's front page is how readers find the shelf, so reaching 1.8 times as many books is worth a handful of readers' hits.

accuracy_vs_reach:
Against the highest-hit systems I gave u

In [14]:
memo = [frontier_choice, accuracy_vs_reach, cold_start, cost, next_step]
assert all(text != "TODO" and any(ch.isdigit() for ch in text) for text in memo)
assert fixed_metrics["hit"] > final_metrics["hit"]  # the named system that beats mine on hit
assert cold_minus_semantic * 4 <= my_cold <= start_cold
assert build_seconds["reranker-full8"] > max(needs.values())
assert full_stack_build > my_build

## Make it yours

### Tune the frontier

Five weightings of my four paths, next to the Part-1 build, the full stack and the 8-path fixed blend.
A row is **dominated** when another row is at least as good on both hit and coverage, and strictly better on one.

In [15]:
weightings = {
    "equal 1/1/1/1": {"item-item": 1.0, "two-tower": 1.0, "feature-tower": 1.0, "semantic": 1.0},
    "semantic 0.5": {"item-item": 1.0, "two-tower": 1.0, "feature-tower": 1.0, "semantic": 0.5},
    "item-item 1.25": {"item-item": 1.25, "two-tower": 1.0, "feature-tower": 1.0, "semantic": 1.0},
    "item-item 1.5 (mine)": my_weights,
    "two-tower 1.25": {"item-item": 1.0, "two-tower": 1.25, "feature-tower": 1.0, "semantic": 1.0},
}
sweep = {}
for label, weights in weightings.items():
    blend = FixedBlendPath(
        {n: system.full8[n] for n in weights}, weights=weights, pool=POOL, name=label
    )
    lists = system.top_lists(blend)
    sweep[label] = {**system.readout(lists), "cold_books": cold_surfaced(lists)}
sweep["part1-build"] = {**part1_metrics, "cold_books": cold_surfaced(part1_lists)}
sweep["full-stack"] = {**full_stack_metrics, "cold_books": full_stack_cold}
sweep["8-path fixed blend"] = {**fixed_metrics, "cold_books": fixed_cold}


def dominated(label):
    a = sweep[label]
    return any(
        b["hit"] >= a["hit"] and b["coverage"] >= a["coverage"]
        and (b["hit"] > a["hit"] or b["coverage"] > a["coverage"])
        for other, b in sweep.items() if other != label
    )


print(f"{'system':<22}{'hit@10':>8}{'coverage':>10}{'Gini':>8}{'cold':>6}  dominated?")
for label, m in sweep.items():
    print(f"{label:<22}{m['hit']:>8.3f}{m['coverage']:>10.3f}{m['exposure_gini']:>8.3f}"
          f"{m['cold_books']:>6}  {'yes' if dominated(label) else 'no'}")

system                  hit@10  coverage    Gini  cold  dominated?
equal 1/1/1/1            0.340     0.318   0.887    56  yes
semantic 0.5             0.350     0.300   0.896    12  yes
item-item 1.25           0.338     0.349   0.877    42  yes
item-item 1.5 (mine)     0.352     0.376   0.867    29  no
two-tower 1.25           0.350     0.288   0.897    35  yes
part1-build              0.306     0.333   0.908     4  yes
full-stack               0.356     0.203   0.913     0  yes
8-path fixed blend       0.362     0.206   0.923    23  no


In [16]:
assert sweep["item-item 1.5 (mine)"] == {**my_metrics, "cold_books": my_cold}
assert not dominated("item-item 1.5 (mine)")
assert dominated("part1-build") and dominated("full-stack")

The sweep shows my weighting is on the frontier, while both the Part-1 build and the full stack are dominated
on hit and coverage (the full stack by the fixed blend it was built on).
The steps between weightings are a few readers each, so I treat the shape, not the third decimal, as the finding.